# Rotation Sensitivity Analysis

Analyse how cp_measure features vary across rotation angles for the 1,600
simulated cells in `rotation_dataset.ome.parquet`.

**Prerequisites:** run `simulate_rotation_sweep.py` first to generate the dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATASET = Path("rotation_dataset.ome.parquet")
assert DATASET.exists(), f"Run simulate_rotation_sweep.py first — {DATASET} not found"

In [ ]:
# Load feature columns only (skip OME-Arrow image/mask structs for speed)
schema = pq.read_schema(DATASET)
meta_cols = ["cell_id", "angle_deg", "aspect_ratio", "y_radius", "stain_type", "stain_corr", "seed"]
feature_cols = [f.name for f in schema if f.name not in meta_cols + ["image", "mask"]]

df = pq.read_table(DATASET, columns=meta_cols + feature_cols).to_pandas()
print(f"Rows: {len(df):,}  |  Feature columns: {len(feature_cols)}")
df.head(3)

## Feature stability across rotation

For each feature, compute the **coefficient of variation (CV)** across the 72
rotation angles, averaged over all 1,600 cells.  Low CV → rotation-stable;
high CV → rotation-sensitive.

In [ ]:
# CV per (cell_id, feature) — std/mean across angles
grp = df.groupby("cell_id")[feature_cols]
cell_cv = grp.std() / grp.mean().abs().replace(0, np.nan)

# Mean CV across all cells
mean_cv = cell_cv.mean().sort_values(ascending=False)

print("Most rotation-SENSITIVE features (highest mean CV):")
print(mean_cv.head(10).to_string())
print()
print("Most rotation-STABLE features (lowest mean CV):")
print(mean_cv.tail(10).to_string())

In [ ]:
# Distribution of CVs
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mean_cv.dropna(), bins=60, edgecolor="none", color="steelblue", alpha=0.8)
ax.axvline(0.05, color="orange", linestyle="--", label="CV = 0.05")
ax.set_xlabel("Mean CV across rotation angles")
ax.set_ylabel("Number of features")
ax.set_title("Distribution of rotation sensitivity (CV) across ~372 features")
ax.legend()
plt.tight_layout()
plt.show()

n_stable = (mean_cv < 0.05).sum()
print(f"Features with CV < 0.05 (essentially rotation-stable): {n_stable}/{len(mean_cv)}")

## Rotation profiles for representative features

In [ ]:
# Pick one cell from each stain type to illustrate profiles
example_cells = (
    df.groupby("stain_type", observed=True).apply(lambda g: g["cell_id"].iloc[0])
    .to_dict()
)

# Top-5 most sensitive and top-5 most stable features
top_sensitive = mean_cv.head(5).index.tolist()
top_stable    = mean_cv.tail(5).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for row_i, (features, title) in enumerate([
    (top_sensitive, "Most sensitive"),
    (top_stable,    "Most stable"),
]):
    for col_i, feat in enumerate(features):
        ax = axes[row_i, col_i]
        for stain_type, cell_id in example_cells.items():
            sub = df[df["cell_id"] == cell_id].sort_values("angle_deg")
            ax.plot(sub["angle_deg"], sub[feat], label=stain_type, linewidth=1)
        ax.set_title(feat[:28], fontsize=8)
        ax.set_xlabel("Angle (°)", fontsize=7)
        if col_i == 0:
            ax.set_ylabel(title, fontsize=8)
        ax.tick_params(labelsize=7)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", fontsize=8, title="stain_type")
plt.suptitle("Feature value vs. rotation angle", y=1.01)
plt.tight_layout()
plt.show()

## Sensitivity by cell parameter

In [ ]:
# Does rotation sensitivity depend on aspect ratio or stain type?
# For each (cell, top-sensitive-feature), compute per-cell CV

top_feat = mean_cv.index[0]  # single most sensitive feature
per_cell = (
    df.groupby(["cell_id", "aspect_ratio", "stain_type", "stain_corr", "y_radius"])[top_feat]
    .agg(cv=lambda x: x.std() / abs(x.mean()) if abs(x.mean()) > 0 else np.nan)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["aspect_ratio", "stain_type"]):
    per_cell.boxplot(column="cv", by=col, ax=ax, grid=False)
    ax.set_title(f"CV of `{top_feat[:30]}` by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("CV")

plt.suptitle("")
plt.tight_layout()
plt.show()